# Data Loading: Dataset, Datafile, and Sample

CREST provides three classes for heterogeneous, multi-source data loading:

- **Datafile** (`crest/data/loading/Datafile.py`) — wraps a single source of data
  (an in-memory `xr.Dataset`, a Zarr/NetCDF path, or an S3 path) and declares how
  windows should be cut from it.
- **Dataset** (`crest/data/loading/Dataset.py`) — collects multiple `Datafile`s and
  generates aligned, multi-source sample windows via `generate_samples`.
- **Sample** (`crest/data/loading/Sample.py`) — a single matched window set: one
  `xr.Dataset` per `Datafile`, all centred on the same coordinate.

This notebook covers:

1. **Synthetic data** — building in-memory `xr.Dataset`s via `synthetic_data`
2. **Datafile and Dataset** — key constructor parameters
3. **Generate samples** — `generate_samples` and its return type
4. **Inspect a sample** — iterating and indexing into a `Sample`
5. **valid_percent** — filtering windows with partial NaN coverage
6. **Batcher integration** — feeding a `Dataset` directly into `Batcher`

In [ ]:
# Allow crest to be imported from the examples/ sub-directory
%cd -q ..

In [ ]:
import logging
import numpy as np
import xarray as xr

from crest.utils.synthetic_data import synthetic_data
from crest.data.loading.Datafile import Datafile
from crest.data.loading.Dataset import Dataset
from crest.data import Batcher

## 1. Synthetic Data

`synthetic_data` (`crest/utils/synthetic_data.py`) generates a list of `xr.Dataset`s
from a list of `data_kwargs` dicts. Each dict specifies:

- **dimensions** — axis names and their length (e.g. `{'x': 9}` → one dimension
  called `x` with 9 elements).
- **chunks** — dask chunk sizes along each dimension.

Here we create three 1-D datasets that will serve as different data sources:
9, 17, and 6 elements along a shared `x` axis.

In [ ]:
data_kwargs = [
    {'dimensions': {'x':  9}, 'chunks': {'x': 3}},   # source 0: 9 elements
    {'dimensions': {'x': 17}, 'chunks': {'x': 6}},   # source 1: 17 elements
    {'dimensions': {'x':  6}, 'chunks': {'x': 2}},   # source 2: 6 elements
]
depths = [{'x': 1}, {'x': 2}, {'x': 0}]

data = synthetic_data(data_kwargs=data_kwargs, verbose=False)

for i, (ds, depth) in enumerate(zip(data, depths)):
    n = ds.sizes['x']
    print(f"Source {i}: {n} elements along x  |  window_depth={depth}")

## 2. Datafile and Dataset

`Datafile` wraps one data source. The key parameters are:

| Parameter | Type | Description |
|---|---|---|
| `location` | `Path \| str \| xr.Dataset` | Data source (path or in-memory dataset) |
| `features` | `list[str]` | Variables to use; default = all |
| `extent` | `dict[str, [lo, hi]]` | Coordinate range filter |
| `window_depth` | `dict[str, int \| (before, after)]` | Elements on each side of a window center |
| `valid_percent` | `dict[str, float]` | Minimum fraction of non-NaN values required (see §5) |

Window depth `{'x': 1}` means 1 element before and 1 after the center → window size 3.
Depth `{'x': 2}` → window size 5. Depth `{'x': 0}` → only the center element.

`Dataset` collects multiple `Datafile`s and handles alignment across their coordinates.

In [ ]:
# One Datafile per xr.Dataset; window_depth controls how large each window is
datafiles = [Datafile(ds, window_depth=depth) for ds, depth in zip(data, depths)]
dataset   = Dataset(datafiles)

print(f"Dataset contains {len(dataset)} Datafile(s)")
print(dataset)

### Inspecting `Datafile.data`

`Datafile.data` exposes the internal `xr.DataArray` that CREST builds from the source
dataset.

In [ ]:
print(datafiles[0].data)

## 3. Generate Samples

`Dataset.generate_samples()` (`crest/data/loading/Dataset.py`) finds all coordinate
locations that are valid across every `Datafile` simultaneously:

1. Each `Datafile` is divided into dask-parallel `Block`s.
2. One `Block` per `Datafile` forms a `Blockset`.
3. Valid windows are matched across `Block`s within each `Blockset`.
4. Results are concatenated into a single lazy dask array of `Sample` objects.

Pass `verbose=False` to suppress the progress log.

In [ ]:
# Dataset.logger resets to INFO on every property access, so use logging.disable
# to silence its output for this demo rather than setting the logger level directly.
logging.disable(logging.INFO)
samples = dataset.generate_samples(verbose=False)
logging.disable(logging.NOTSET)

print(f"Found {samples.shape[0]} samples")
print(f"Dask array dtype names: {samples.dtype.names}")

## 4. Inspect a Sample

`samples` is a lazy dask array. Indexing with `.compute()` materialises one element
into a `Sample` object. A `Sample` contains one `xr.Dataset` window per `Datafile`;
iterating over it yields those windows in order.

For our three `Datafile`s with depths `[1, 2, 0]`, each sample window has sizes
`[3, 5, 1]` along the `x` axis.

In [ ]:
# Materialise the first sample
sample = samples[0].compute()

print(f"Sample contains {len(sample)} window(s) — one per Datafile\n")

for i, window in enumerate(sample):
    x_vals = window['var_0'].values
    print(f"  Datafile {i}  |  depth={depths[i]}  |  shape={x_vals.shape}  |  values={x_vals}")

## 5. `valid_percent`

By default (`valid_percent={}`), every element of a window must be non-NaN. The
`valid_percent` parameter relaxes this constraint: it maps dimension names to the
minimum fraction of non-NaN values required for a window to be considered valid.

For example, `valid_percent={'x': 0.5}` accepts windows where at least half of the
`x` values are non-NaN — useful when a sensor occasionally drops readings.

Below we create a 9-element dataset with one NaN at position 3. With `window_depth=1`
each window has 3 elements. Centers near the NaN will have 1 missing value (33% NaN).

| Setting | Windows accepted | Count |
|---|---|---|
| Default (100% valid) | Only fully clean windows | 4 |
| `valid_percent={'x': 0.5}` | Windows ≥ 50% clean (1 NaN tolerated) | 7 |

In [ ]:
# 9-element dataset with a NaN at position 3
vals_with_nan = np.array([1., 2., 3., np.nan, 5., 6., 7., 8., 9.], dtype='float32')
ds_nan = xr.Dataset(
    {'var_0': ('x', vals_with_nan)},
    coords={'x': np.arange(9, dtype='float32')},
)

logging.disable(logging.INFO)

# Default: all elements must be valid (valid_percent={})
samples_strict  = Dataset([Datafile(ds_nan, window_depth={'x': 1})]).generate_samples(verbose=False)

# Relaxed: ≥ 50% of window elements must be valid
samples_relaxed = Dataset([Datafile(ds_nan, window_depth={'x': 1},
                                    valid_percent={'x': 0.5})]).generate_samples(verbose=False)

logging.disable(logging.NOTSET)

print(f"Default   (100% valid): {samples_strict.shape[0]:2d} samples")
print(f"Relaxed   ( 50% valid): {samples_relaxed.shape[0]:2d} samples")

## 6. Batcher Integration

`Batcher` (`crest/data/batching/Batcher.py`) accepts a `Dataset` directly. When
`features=None` (the default for heterogeneous `Dataset` inputs), each batch is a
numpy array of `Sample` objects — one element per sample in the batch.

This is the entry point for ML training loops: the model processes each `Sample` by
reading its per-`Datafile` windows and assembling the inputs and targets.

In [ ]:
# Create a Batcher over the original 3-Datafile dataset
batcher = Batcher(dataset, batch_size=5, workers=0, repeat=False, shuffle=False)

with batcher as b:
    batch = next(b)

print(f"Batch type  : {type(batch).__name__}")
print(f"Batch length: {len(batch)}  (= batch_size)")
print(f"Element type: {type(batch[0]).__name__}")

# Inspect the first sample in the batch
s0 = batch[0]
print(f"\nFirst sample — {len(s0)} window(s):")
for i, window in enumerate(s0):
    print(f"  Datafile {i}: shape={window['var_0'].shape}  "
          f"values={window['var_0'].values}")

In [ ]:
# Count total batches produced from 15 samples with batch_size=5
count = 0
with Batcher(dataset, batch_size=5, workers=0, repeat=False, shuffle=False) as b:
    for _ in b:
        count += 1
print(f"{count} batches × 5 samples/batch = {count * 5} samples total")